# Recurrent Neural Networks (RNNs): From Start to Finish

Welcome to sequence modeling. In this tutorial, we will build a basic Recurrent Neural Network. 

Unlike standard neural networks that use a different set of weights for every layer, an RNN applies the **exact same weights** across every time step in a sequence. This allows the network to process sequences of any length while keeping the parameter count low.

### The Mathematical Core: The Hidden State Update
At any given time step $t$, the RNN receives the current input $x_t$ and the previous hidden state $h_{t-1}$. It combines them using two weight matrices ($W_{xh}$ and $W_{hh}$) and an activation function (usually $\tanh$) to produce the new hidden state $h_t$:

$$h_t = \tanh(W_{xh} x_t + b_{xh} + W_{hh} h_{t-1} + b_{hh})$$

### Our Analytical Pipeline:
1. **Environment Setup:** Importing PyTorch.
2. **Sequential Data Generation:** Synthesizing a decaying time-series signal.
3. **Architecture Design:** Building the RNN class and analyzing its tensor dimensions.
4. **Training:** Backpropagation Through Time (BPTT).
5. **Evaluation:** Analyzing the model's ability to extrapolate sequential patterns.

In [ ]:
# Cell 2: Environment Setup and Imports
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Set seeds for analytical reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch Version: {torch.__version__}")
print("Libraries imported successfully. Ready for Sequence Modeling.")

## Step 1: Generating Sequential Data

To train an RNN, we need data where the chronological order matters. We will generate a **Damped Sine Wave** (a wave whose amplitude decreases over time, simulating a spring losing energy or a decaying acoustic signal).

To feed this into an RNN, we must restructure the 1D wave into a **Sliding Window** dataset. If our window size is $W$, the features ($X$) are the points $[t_1 \dots t_W]$, and the target ($y$) is the very next point $t_{W+1}$.

In [ ]:
# Cell 4: Data Generation and Windowing

# 1. Generate a noisy damped sine wave
time_steps = np.linspace(0, 100, 1000)
# Formula: sin(t) * e^(-0.02 * t) + noise
damped_wave = np.sin(time_steps) * np.exp(-0.02 * time_steps) + np.random.normal(0, 0.02, 1000)

# 2. Define the sliding window function
def create_sequences(data, window_size):
    sequences = []
    targets = []
    for i in range(len(data) - window_size):
        seq = data[i:i + window_size]
        label = data[i + window_size]
        sequences.append(seq)
        targets.append(label)
    return np.array(sequences), np.array(targets)

window_size = 20
X, y = create_sequences(damped_wave, window_size)

# 3. Convert to PyTorch Tensors
# RNNs expect strictly formatted 3D tensors: (Batch Size, Sequence Length, Input Features)
# Since we have 1 variable (amplitude) at each time step, Input Features = 1.
X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(-1) 
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

# Split into Training (80%) and Testing (20%)
split_idx = int(len(X) * 0.8)
X_train, X_test = X_tensor[:split_idx], X_tensor[split_idx:]
y_train, y_test = y_tensor[:split_idx], y_tensor[split_idx:]

print("--- Analytical Tensor Shapes ---")
print(f"Training Input Shape:  {X_train.shape} -> (Batch: {X_train.shape[0]}, Seq_Len: {X_train.shape[1]}, Input_Dim: {X_train.shape[2]})")
print(f"Training Target Shape: {y_train.shape} -> (Batch: {y_train.shape[0]}, Output_Dim: {y_train.shape[1]})")

## Step 2: Building the RNN Architecture

In PyTorch, we build neural networks by inheriting from `nn.Module`.

We will use the highly optimized `nn.RNN` layer. The key analytical concept here is that the `nn.RNN` layer processes the entire sequence and outputs the hidden state for *every single time step*. 
Since our goal is to predict just one number at the end of the sequence, we only care about the **final hidden state** ($h_{final}$). We will pass that final state through a Linear (Dense) layer to shrink the multi-dimensional memory down to a single continuous prediction.

In [ ]:
# Cell 6: Defining the RNN Model

class SimpleRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super(SimpleRNN, self).__init__()
        
        self.hidden_size = hidden_size
        
        # 1. The Core RNN Layer
        # batch_first=True tells PyTorch that the first dimension of our tensor is the batch size.
        # nonlinearity='tanh' is the default, providing the squashing function for the hidden state.
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True, nonlinearity='tanh')
        
        # 2. The Final Output Layer
        # Maps the complex hidden state back down to a single numeric prediction
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # The RNN returns two tensors:
        # 'out': The hidden states for EVERY time step -> Shape: (Batch, Seq, Hidden_Size)
        # 'h_n': The final hidden state of the sequence -> Shape: (Num_Layers, Batch, Hidden_Size)
        out, h_n = self.rnn(x)
        
        # We slice the 'out' tensor to extract the hidden state from the very last time step
        final_hidden_state = out[:, -1, :]
        
        # Pass the final hidden state through the linear layer to get the prediction
        prediction = self.linear(final_hidden_state)
        return prediction

# Instantiate the model
model = SimpleRNN()
print(model)

## Step 3: Loss, Optimizer, and Backpropagation Through Time

Because we are predicting a continuous value (Amplitude), we use **Mean Squared Error (MSE)** as our Loss Function. We use the **Adam** optimizer to perform Gradient Descent.

Training an RNN requires a specialized calculus called **Backpropagation Through Time (BPTT)**. Because the network reuses the exact same weights at every time step, calculating the gradient for the weights requires rolling the calculus chain rule backward through every previous time step in the sequence window. PyTorch's `loss.backward()` handles this unrolling graph automatically.

In [ ]:
# Cell 8: The Training Loop

# Hyperparameters
epochs = 200
learning_rate = 0.01

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("--- Starting Training (BPTT) ---")
loss_history = []

for epoch in range(epochs):
    # 1. Forward Pass
    predictions = model(X_train)
    
    # 2. Calculate Mathematical Error
    loss = criterion(predictions, y_train)
    
    # 3. Backpropagation Through Time
    optimizer.zero_grad() # Clear old gradients
    loss.backward()       # Calculate new gradients by unrolling the sequence backwards
    optimizer.step()      # Update the shared weights
    
    loss_history.append(loss.item())
    
    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | MSE Loss: {loss.item():.6f}")

print("Training Complete!")

## Step 4: Analytical Evaluation and Visualization

How well did the RNN learn the physics of a decaying wave? 

We will feed the model our completely unseen `X_test` data. This is challenging because the wave in the test set has a much lower amplitude than the wave in the training set (due to the exponential decay). The RNN must prove it learned the *pattern* of decay, not just memorized the raw values.

In [ ]:
# Cell 10: Evaluation and Plotting

# Turn off gradient tracking for evaluation to save memory and compute
model.eval()
with torch.no_grad():
    # Make predictions on the test set
    test_predictions = model(X_test).numpy()
    
# Calculate final test error
test_loss = criterion(torch.tensor(test_predictions), y_test).item()
print(f"Final Test MSE Loss: {test_loss:.6f}\n")

# Realign the arrays for plotting so they match up chronologically
time_test = time_steps[split_idx + window_size:]
true_values = y_test.numpy()

# Visualizing the Results
plt.figure(figsize=(12, 5))
plt.plot(time_test, true_values, label="True Ground Truth", color="teal", linewidth=2, alpha=0.6)
plt.plot(time_test, test_predictions, label="RNN Predictions", color="orange", linestyle="--", linewidth=2)

plt.title("RNN Temporal Prediction on Unseen Decaying Wave", fontsize=15)
plt.xlabel("Time Steps", fontsize=12)
plt.ylabel("Amplitude", fontsize=12)
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.show()

print("Analytical Conclusion: The RNN successfully updated its hidden state over the 20-step window to recognize the current phase and amplitude of the wave, closely tracking the ground truth!")

## Conclusion

You have successfully implemented a Recurrent Neural Network!

**Key Analytical Takeaways:**
1. **Shared Weights:** Unlike MLPs that have separate weights for every feature, an RNN applies the exact same $W_{xh}$ and $W_{hh}$ matrices at step 1 as it does at step 20. This allows it to generalize across sequence lengths.
2. **The Hidden State:** This vector acts as the network's short-term memory, summarizing all preceding context required to make a prediction.
3. **The Fatal Flaw (Vanishing Gradients):** Notice we only used a window size of 20. If we used a window size of 500, the gradients during BPTT would multiply by themselves 500 times. Because they are often fractions (due to the `tanh` derivative), the gradient shrinks to 0. The network stops learning and "forgets" the start of the sequence. 

This mathematical bottleneck is exactly why **LSTMs (Long Short-Term Memory)** networks were invented, introducing distinct "Gates" to protect the gradient over long distances!